# FlashInspector AI – Train on Google Colab (GPU)

Use this notebook to train the fire safety YOLO model on Colab's free GPU so you don't need a strong local GPU.

**Before running:**
1. In Colab: **Runtime → Change runtime type → T4 GPU** (or better).
2. Have your [Roboflow API key](https://app.roboflow.com/settings/api) ready.
3. Either push this repo to GitHub and set `REPO_URL` below, or upload the `flashinspector-ai` folder to Colab and skip the clone cell.

## 1. Check GPU

In [ ]:
!nvidia-smi 2>/dev/null || echo "No GPU (nvidia-smi not found). In Colab: Runtime → Change runtime type → T4 GPU."

## 2. Get the project code

**Option A – Clone from GitHub:** Set `REPO_URL` to your repo in the next cell and run it (leave `USE_UPLOADED = False`).

**Option B – Folder already in Colab:** If `flashinspector-ai` is already at a path (e.g. you unzipped it), set `USE_UPLOADED = True` and `UPLOADED_PATH` in the cell below and run.

**Option C – Upload zip (no GitHub):** Run the **"Optional - Option C"** cell below first: upload a zip of your `flashinspector-ai` folder. Then in the next cell set `USE_UPLOADED = True`, `UPLOADED_PATH = "/content/flashinspector-ai"` (or the path that appeared after unzip) and run.

In [ ]:
# OPTIONAL - Option C: Upload a zip of flashinspector-ai (no GitHub needed)
# 1. On your computer: zip the flashinspector-ai folder
# 2. Run this cell, click "Choose Files", select the zip
# 3. In the cell below, set USE_UPLOADED = True and run it

from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # opens file picker
for name in uploaded:
    if name.endswith(".zip"):
        Path("/content").mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(name, "r") as z:
            z.extractall("/content")
        print(f"Unzipped to /content (look for flashinspector-ai folder)")
        break

In [ ]:
# Choose ONE option:
# A) Clone from GitHub (set REPO_URL below)
# B) You already have the folder at UPLOADED_PATH (e.g. after unzipping)
# C) Upload a zip: run the next cell first to upload flashinspector-ai.zip, then set USE_UPLOADED = True

REPO_URL = "https://github.com/yourusername/fire.git"  # for Option A: your repo URL
USE_UPLOADED = False  # True = use folder at UPLOADED_PATH (Options B or C)
UPLOADED_PATH = "/content/flashinspector-ai"

if USE_UPLOADED:
    import os
    PROJECT_DIR = UPLOADED_PATH
    assert os.path.isdir(PROJECT_DIR), f"Folder not found: {UPLOADED_PATH}. For Option C, run the upload cell below first."
    %cd $PROJECT_DIR
    print(f"Using project at {PROJECT_DIR}")
else:
    !git clone --depth 1 $REPO_URL /content/fire
    %cd /content/fire/flashinspector-ai
    PROJECT_DIR = "/content/fire/flashinspector-ai"
    print(f"Cloned and using {PROJECT_DIR}")

## 3. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow opencv-python python-dotenv pyyaml tqdm

## 4. Set your Roboflow API key

Paste your key below (get it from https://app.roboflow.com/settings/api). It stays in this session only.

In [ ]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Paste your Roboflow API key: ")

## 5. Download dataset

Downloads the dataset you want to train on. Change `--dataset` if needed (`fire_extinguisher`, `fire_smoke`, `emergency_exit`, `construction_safety`).

In [ ]:
!python download_datasets.py --dataset fire_extinguisher

## 6. Train on Colab GPU

Uses the Colab GPU automatically. Colab-friendly defaults:
- `--batch 8` to avoid OOM on free T4 (increase to 16 if you have more memory).
- `--epochs 50` to finish in one session (increase if you have Colab Pro).
- `--size nano` is fastest; use `small` or `medium` if you want better accuracy.

Edit the command below to change dataset, size, or epochs.

In [ ]:
!python fire_safety_datasets/train_model.py \
  --dataset fire_extinguisher \
  --size nano \
  --epochs 50 \
  --batch 8 \
  --no-export

## 7. Save model to Google Drive (recommended)

Colab disk is wiped when the runtime disconnects. Mount Drive and copy the best weights so you keep them.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

# Where to save on your Drive
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/flashinspector_models")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_pt = Path("fire_safety_models/fire_extinguisher_nano/weights/best.pt")
if best_pt.exists():
    dest = DRIVE_SAVE_DIR / best_pt.name
    shutil.copy(best_pt, dest)
    print(f"Saved to Google Drive: {dest}")
else:
    print("best.pt not found. Check the run name (e.g. fire_extinguisher_nano) and path above.")
    !ls -la fire_safety_models/

## 8. Test the model in Colab

Run inference on an image, video, or YouTube link and see the result in the notebook.

- **Same session:** If you just trained above, the model is already in `fire_safety_models/`.
- **New session:** Mount Drive and point to your saved `best.pt` in `My Drive/flashinspector_models/`.

**Input options:** Upload an image/video (next cell), use a YouTube URL (cell after that), or skip both to use a dataset sample (after download + train).

In [ ]:
# Model path: use local weights if you just trained, else load from Drive
from pathlib import Path
import os

# Ensure we're in the project dir (needed if you run section 8 without running section 2 first)
for d in [".", "/content/fire/flashinspector-ai", "/content/flashinspector-ai"]:
    if (Path(d) / "fire_safety_datasets").exists():
        os.chdir(Path(d).resolve())
        print(f"Working dir: {os.getcwd()}")
        break

MODEL_PATH = Path("fire_safety_models/fire_extinguisher_nano/weights/best.pt")
if not MODEL_PATH.exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_dir = Path("/content/drive/MyDrive/flashinspector_models")
    MODEL_PATH = drive_dir / "best.pt"
    if not MODEL_PATH.exists():
        # Search for best.pt in subfolders (e.g. fire_extinguisher_nano/weights/)
        found = list(drive_dir.rglob("best.pt"))
        MODEL_PATH = found[0] if found else None
    if MODEL_PATH is None or not Path(MODEL_PATH).exists():
        raise FileNotFoundError("No best.pt found in Drive/flashinspector_models. Train first (section 6) and save (section 7).")
print(f"Using model: {MODEL_PATH}")

In [ ]:
# Upload an image or video to test (or skip and use a dataset image in the next cell)
from google.colab import files
uploaded = files.upload()  # click "Choose Files" and pick an image or video
TEST_IMAGE = list(uploaded.keys())[0] if uploaded else None
if TEST_IMAGE:
    print(f"Uploaded: {TEST_IMAGE}")

In [ ]:
# OR: Use a YouTube URL (set YOUTUBE_URL and run; skip if you already uploaded above)
YOUTUBE_URL = ""  # e.g. "https://www.youtube.com/watch?v=58naKHqpCWo"

if YOUTUBE_URL:
    import re
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"], check=True)
    import yt_dlp
    # Unique filename per video so a new link always downloads fresh (no stale file)
    video_id = re.search(r"(?:v=|/)([a-zA-Z0-9_-]{11})(?:\?|&|$)", YOUTUBE_URL)
    video_id = video_id.group(1) if video_id else "video"
    youtube_out = Path.cwd() / f"youtube_video_{video_id}.mp4"
    ydl_opts = {"format": "best[ext=mp4]/best", "outtmpl": str(youtube_out), "quiet": False}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YOUTUBE_URL])
    TEST_IMAGE = str(youtube_out)
    print(f"Downloaded to: {TEST_IMAGE}")

In [ ]:
# If you didn't upload an image, use a sample from the dataset (run download + train first)
try:
    _ = TEST_IMAGE
except NameError:
    TEST_IMAGE = None
if not TEST_IMAGE:
    sample_dir = Path("fire_safety_datasets/fire_extinguisher/valid/images")
    if sample_dir.exists():
        samples = list(sample_dir.glob("*.*"))[:1]
        TEST_IMAGE = str(samples[0]) if samples else None
    if not TEST_IMAGE:
        raise SystemExit("Upload an image in the cell above, or run section 5 (download dataset) first.")

# Find project dir (where test_model.py lives) - cwd can be /content in some sessions
PROJECT_DIR = None
for d in [Path.cwd(), Path("/content/fire/flashinspector-ai"), Path("/content/flashinspector-ai")]:
    script = d / "fire_safety_datasets" / "test_model.py"
    if script.exists():
        PROJECT_DIR = d
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Run section 2 (get project code) first. test_model.py not found.")

# Resolve path - uploaded/YouTube files can be in cwd or /content/
test_path = Path(TEST_IMAGE)
if not test_path.is_absolute():
    for base in [Path.cwd(), Path("/content"), PROJECT_DIR]:
        p = (base / test_path).resolve()
        if p.exists():
            test_path = p
            break
if not test_path.exists():
    raise FileNotFoundError(f"Image/video not found: {test_path}. Re-run the upload/YouTube cell above.")
TEST_IMAGE = str(test_path)
print(f"Project: {PROJECT_DIR}")
print(f"Model: {MODEL_PATH} (exists: {Path(MODEL_PATH).exists()})")
print(f"Input: {TEST_IMAGE} (exists: {Path(TEST_IMAGE).exists()})")

# Run inference (direct for images; script for video)
from ultralytics import YOLO
import cv2
import subprocess

out_dir = PROJECT_DIR / "inference_results"
out_dir.mkdir(exist_ok=True)
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

if Path(test_path).suffix.lower() in VIDEO_EXTS:
    # Video (including YouTube downloads): use script
    script_path = PROJECT_DIR / "fire_safety_datasets" / "test_model.py"
    r = subprocess.run(
        ["python", str(script_path), TEST_IMAGE, "--model", str(MODEL_PATH), "--conf", "0.25", "--frame-skip", "5"],
        cwd=str(PROJECT_DIR), capture_output=True, text=True
    )
    print(r.stdout)
    if r.stderr:
        print("STDERR:", r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"test_model.py failed (code {r.returncode}). Check output above.")
else:
    # Image: run directly (avoids shell path issues)
    model = YOLO(str(MODEL_PATH))
    results = model(test_path, conf=0.25, verbose=True)[0]
    out_path = out_dir / f"result_{Path(test_path).name}"
    cv2.imwrite(str(out_path), annotated := results.plot())
    print(f"Saved to: {out_path}")

In [ ]:
# Show the result in the notebook (image = display; video = download link)
from IPython.display import Image, display
from pathlib import Path
from google.colab import files as colab_files

# Results are in project dir (PROJECT_DIR set by inference cell)
result_dir = PROJECT_DIR / "inference_results" if "PROJECT_DIR" in dir() else Path.cwd() / "inference_results"
name = Path(TEST_IMAGE).name
stem = Path(TEST_IMAGE).stem
result_path = result_dir / f"result_{name}"
if not result_path.exists():
    result_path = result_dir / f"result_{stem}.mp4"
if result_path.exists():
    if result_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"):
        display(Image(str(result_path), width=600))
    else:
        print(f"Annotated video saved: {result_path}")
        colab_files.download(str(result_path))
    print(f"Output: {result_path}")
else:
    print("Result not found. If the cell above showed an error, fix it and re-run.")
    print(f"Looked in: {result_dir}")
    if result_dir.exists():
        !ls -la "{result_dir}"
    else:
        print("inference_results/ was not created – test_model.py likely failed. Run sections 2 and 3 first.")